## 1. Setup & Imports
All third-party imports, device selection, and global random seed.

In [ ]:
import os, random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

# Install POT if needed
try:
    import ot
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "POT", "-q"])
    import ot

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")


## 2. Configuration
All hyperparameters and paths in one place (change is only done here).

In [ ]:
cfg = SimpleNamespace(
    # --- Paths ---
    dataset_dir="/kaggle/input/competitions/csiro-biomass",
    dino_weights_dir="/kaggle/input/datasets/mayaevensen/dinov2-weights/dinov2-small",
    output_dir="/kaggle/working",
    # --- Model ---
    input_dim=384,
    latent_dim=64,
    output_dim=5,
    dropout=0.1,
    # --- Training ---
    epochs=80,
    lr=3e-4,
    weight_decay=1e-3,
    batch_size=64,
    seed=42,
    # --- Split ---
    n_splits=5,
    val_fold=0,
    # --- CHARMS ---
    n_clusters=40, # K-Means groups for image feature dimensions
    tab_embed_dim=32, # FT-Transformer per-attribute embedding dim
    ft_n_blocks=2, # FT-Transformer number of transformer blocks
    li2t_weight=0.1, # Weight of Li2t auxiliary loss
    tab_loss_weight=0.5, # Weight of tabular-model regression loss
    ot_update_freq=5, # Refresh OT transfer matrix every N epochs
    ot_subsample=512, # Max samples used per OT cost-matrix computation
)

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1,
           "GDM_g": 0.2, "Dry_Total_g": 0.5}

TRAIN_CSV     = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV      = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

print("train.csv:",  os.path.exists(TRAIN_CSV))
print("test.csv:",   os.path.exists(TEST_CSV))
print("train dir:",  os.path.exists(TRAIN_IMG_DIR))
print("test dir:",   os.path.exists(TEST_IMG_DIR))
print("dino dir:",   os.path.exists(cfg.dino_weights_dir))


## 3. Load DINOv2
Load ViT-S/14 from uploaded HuggingFace weights; smoke-test at 504×252 resolution.

In [ ]:
print(f"Loading DINOv2 ViT-S/14 from {cfg.dino_weights_dir} ...")
dino = AutoModel.from_pretrained(cfg.dino_weights_dir).eval().to(device)
for p in dino.parameters():
    p.requires_grad_(False)

_dummy = torch.zeros(1, 3, 252, 504, device=device)
with torch.no_grad():
    _out = dino(pixel_values=_dummy, interpolate_pos_encoding=True)
    _cls = _out.last_hidden_state[:, 0, :]
assert _cls.shape == (1, 384), f"Expected (1, 384), got {_cls.shape}"
print(f"DINOv2 smoke test passed - CLS shape: {_cls.shape}")
del _dummy, _out, _cls

Loading DINOv2 ViT-S/14 from /kaggle/input/datasets/mayaevensen/dinov2-weights/dinov2-small ...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 smoke test passed — CLS shape: torch.Size([1, 384])


## 4. Load Training CSV
Pivot long-format CSV to wide format (one row per image) and extract Y matrix.

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["image_id"] = df_raw["sample_id"].str.split("__").str[0]

df_wide = df_raw.pivot_table(
    index=["image_id", "image_path"],
    columns="target_name",
    values="target",
).reset_index()

Y_all = df_wide[TARGETS].values.astype(np.float32) # (N, 5)
train_image_ids_all = df_wide["image_id"].values

print(f"Training images: {len(df_wide)}")
print(f"Y_all shape:     {Y_all.shape}")
print(df_wide.head(3))

Training images: 357
Y_all shape:     (357, 5)
target_name      image_id              image_path  Dry_Clover_g  Dry_Dead_g  \
0            ID1011485656  train/ID1011485656.jpg          0.00     31.9984   
1            ID1012260530  train/ID1012260530.jpg          0.00      0.0000   
2            ID1025234388  train/ID1025234388.jpg          6.05      0.0000   

target_name  Dry_Green_g  Dry_Total_g   GDM_g  
0                16.2751      48.2735  16.275  
1                 7.6000       7.6000   7.600  
2                 0.0000       6.0500   6.050  


## 4b. CHARMS - Tabular Feature Preparation
Extract numerical and categorical attributes from `train.csv` aligned to the
pivoted image order. These are used **only during training** (not at test time).

| Attribute | Type | Notes |
|---|---|---|
| `Pre_GSHH_NDVI` | numerical | z-score normalised |
| `Height_Ave_cm` | numerical | z-score normalised |
| `State` | categorical | Australian state label-encoded |
| `Species_first` | categorical | first species token, label-encoded |

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

TAB_NUM_COLS = ["Pre_GSHH_NDVI", "Height_Ave_cm"]

# One row per image_id, keeping the tabular side-channel columns
df_tab = df_raw.drop_duplicates("image_id").copy()
df_tab["Species_first"] = df_tab["Species"].str.split("_").str[0].fillna("Unknown")
df_tab = df_tab.set_index("image_id")

# Align to df_wide row order
df_tab_aligned = df_tab.loc[df_wide["image_id"].values]

# --- Numerical features: z-score ---
tab_num_scaler = StandardScaler()
X_tab_num_base = tab_num_scaler.fit_transform(
    df_tab_aligned[TAB_NUM_COLS].fillna(0).values.astype(np.float32)
)  # (N_orig, 2)

# --- Categorical features: label-encode ---
state_le = LabelEncoder()
X_state_base = state_le.fit_transform(
    df_tab_aligned["State"].fillna("Unknown").values
).astype(np.int64)  # (N_orig,)

species_le = LabelEncoder()
X_species_base = species_le.fit_transform(
    df_tab_aligned["Species_first"].values
).astype(np.int64)  # (N_orig,)

X_tab_cat_base = np.stack([X_state_base, X_species_base], axis=1)  # (N_orig, 2)

N_NUM_ATTRS   = len(TAB_NUM_COLS) # 2
N_CAT_CLASSES = [len(state_le.classes_), # cardinality per categorical
                 len(species_le.classes_)]
N_CAT_ATTRS   = len(N_CAT_CLASSES) # 2
D_TAB         = N_NUM_ATTRS + N_CAT_ATTRS # 4 total attributes

# Repeat ×4 to match the 4-orientation augmented dataset
X_tab_num_all = np.repeat(X_tab_num_base, 4, axis=0) # (N*4, 2)
X_tab_cat_all = np.repeat(X_tab_cat_base, 4, axis=0) # (N*4, 2)

print(f"Tabular attributes: {N_NUM_ATTRS} numerical, {N_CAT_ATTRS} categorical (D={D_TAB})")
print(f"  State cardinality : {N_CAT_CLASSES[0]}")
print(f"  Species cardinality: {N_CAT_CLASSES[1]}")
print(f"X_tab_num_all: {X_tab_num_all.shape}  X_tab_cat_all: {X_tab_cat_all.shape}")

Tabular attributes: 2 numerical, 2 categorical (D=4)
  State cardinality : 4
  Species cardinality: 9
X_tab_num_all: (1428, 2)  X_tab_cat_all: (1428, 2)


## 5. Extract DINOv2 Features - Training Images
Preprocess at 504×252 and extract CLS tokens with 4-orientation augmentation (identity, hflip, vflip, hflip+vflip).

In [ ]:
_dino_transform = T.Compose([
    T.Resize((252, 504)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_features(image_paths, model, transform):
    """Return (N, 384) float32 array of CLS-token features."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(pixel_values=x, interpolate_pos_encoding=True)
            feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
        feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


_ORIENTATIONS = [
    lambda img: img,
    lambda img: TF.hflip(img),
    lambda img: TF.vflip(img),
    lambda img: TF.hflip(TF.vflip(img)),
]


def extract_features_augmented(image_paths, model, transform):
    """Return (N*4, 384) array - each image appears 4 times (id, hflip, vflip, hflip+vflip)."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        for flip_fn in _ORIENTATIONS:
            x = transform(flip_fn(img)).unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(pixel_values=x, interpolate_pos_encoding=True)
                feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
            feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


train_image_paths = [
    os.path.join(TRAIN_IMG_DIR, f"{iid}.jpg")
    for iid in train_image_ids_all
]
N_orig = len(train_image_paths)
print(f"Extracting features for {N_orig} training images (4 orientations each)...")
X_all = extract_features_augmented(train_image_paths, dino, _dino_transform)

Y_all               = np.repeat(Y_all,               4, axis=0)
train_image_ids_all = np.repeat(train_image_ids_all, 4)
image_group_ids     = np.repeat(np.arange(N_orig),   4)

print(f"X_all shape:         {X_all.shape}")
print(f"Y_all shape:         {Y_all.shape}")
print(f"Unique groups:       {len(np.unique(image_group_ids))}")

assert X_all.shape == (N_orig * 4, 384)
assert Y_all.shape == (N_orig * 4, 5)
assert len(image_group_ids) == N_orig * 4
assert len(np.unique(image_group_ids)) == N_orig
assert np.all(np.bincount(image_group_ids) == 4)
print("Sanity checks passed.")

Extracting features for 357 training images (4 orientations each)...
  50/357
  100/357
  150/357
  200/357
  250/357
  300/357
  350/357
X_all shape:         (1428, 384)
Y_all shape:         (1428, 5)
Unique groups:       357
Sanity checks passed.


## 6. Extract DINOv2 Features - Test Images
Same preprocessing pipeline applied to the held-out test set.

In [7]:
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw["image_id"] = df_test_raw["sample_id"].str.split("__").str[0]
df_test_unique = df_test_raw.drop_duplicates("image_id").copy()

test_image_ids = df_test_unique["image_id"].values
test_image_paths = [
    os.path.join(TEST_IMG_DIR, f"{iid}.jpg")
    for iid in test_image_ids
]

print(f"Extracting features for {len(test_image_paths)} test images...")
X_test = extract_features(test_image_paths, dino, _dino_transform)
print(f"X_test shape: {X_test.shape}")

Extracting features for 1 test images...
X_test shape: (1, 384)


## 7. GroupKFold Train/Val Split
Fold 0 of a 5-fold GroupKFold; groups on original image index so all 4 orientations of an image land in the same fold.

In [ ]:
gkf = GroupKFold(n_splits=cfg.n_splits)
splits = list(gkf.split(X_all, groups=image_group_ids))
train_idx, val_idx = splits[cfg.val_fold]

X_train = X_all[train_idx]
Y_train_raw = Y_all[train_idx]
X_val = X_all[val_idx]
Y_val_raw = Y_all[val_idx]

train_groups = set(image_group_ids[train_idx].tolist())
val_groups = set(image_group_ids[val_idx].tolist())
assert train_groups.isdisjoint(val_groups), "Group leakage!"
print("Group overlap check: PASSED (zero overlap)")

print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print(f"Y_train range: [{Y_train_raw.min():.1f}, {Y_train_raw.max():.1f}]")

Group overlap check: PASSED (zero overlap)
Train: (1140, 384)  Val: (288, 384)
Y_train range: [0.0, 185.7]


## 7b. CHARMS - Split Tabular Features & K-Means Channel Clustering

**Channel clustering rationale**: CHARMS groups image channels that share similar
co-activation patterns before computing the OT cost matrix. The paper operates
on CNN spatial feature maps (H×W per channel); here we adapt to DINOv2 CLS tokens
by treating the 384 feature *dimensions* as channels. Each dimension yields an
N-dimensional co-activation vector across the training set; K-Means on these 384
vectors produces C′=40 semantically coherent groups.

> **Incompatibility note**: the paper's channels have H×W spatial values per sample,
> giving rich pairwise similarity matrices. DINOv2 CLS tokens have one scalar per
> dimension per sample, so we cluster dimensions (not spatial locations) to retain
> non-trivial within-cluster cosine similarities. The math is otherwise identical.

In [ ]:
from sklearn.cluster import KMeans

# --- Split tabular features using the same train/val indices ---
X_tab_num_train = X_tab_num_all[train_idx]
X_tab_cat_train = X_tab_cat_all[train_idx]
X_tab_num_val = X_tab_num_all[val_idx]
X_tab_cat_val = X_tab_cat_all[val_idx]


def compute_channel_clusters(X_img: np.ndarray, n_clusters: int) -> np.ndarray:
    """
    Cluster the C feature dimensions using K-Means on their cross-sample
    co-activation vectors.

    X_img : (N, C)  pre-extracted image features
    Returns (C,) int array - cluster label for each feature dimension.
    """
    channel_vecs = X_img.T # (C, N): each row = one dimension's activations
    norms = np.linalg.norm(channel_vecs, axis=1, keepdims=True) + 1e-8
    channel_vecs_n = channel_vecs / norms
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    return km.fit_predict(channel_vecs_n).astype(np.int32)  # (C,)


print(f"Fitting K-Means (k={cfg.n_clusters}) on {X_train.shape[1]} feature dimensions...")
channel_cluster_labels = compute_channel_clusters(X_train, cfg.n_clusters)

counts = np.bincount(channel_cluster_labels)
print(f"  Cluster sizes - min: {counts.min()}, max: {counts.max()}, "
      f"mean: {counts.mean():.1f}")
print("Channel clustering done.")

Fitting K-Means (k=40) on 384 feature dimensions...
  Cluster sizes — min: 1, max: 39, mean: 9.6
Channel clustering done.


## 8. Model Architecture - Encoder + MLP Head
Encoder 384→256→64, Head 64→32→5.

In [ ]:
class Encoder(nn.Module):
    """384 → 256 → 64 (GELU, Dropout)."""

    def __init__(self, input_dim=384, latent_dim=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class Head(nn.Module):
    """64 → 32 → 5 (GELU, Dropout)."""

    def __init__(self, latent_dim=64, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class BiomassModel(nn.Module):

    def __init__(self, input_dim=384, latent_dim=64, output_dim=5, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head = Head(latent_dim, output_dim, dropout)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.head(self.encode(x))


print("BiomassModel defined.")
_tmp = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout)
n_params = sum(p.numel() for p in _tmp.parameters())
print(f"Parameters: {n_params:,}")
del _tmp


## 8b. CHARMS Modules

Three components added on top of the baseline architecture:

1. **`FTTransformer`** - encodes tabular attributes into per-attribute embeddings
   (for OT alignment) and predicts biomass targets (for the tabular auxiliary loss).
2. **`Li2tHeads`** - attribute prediction heads: MSE for numerical, cross-entropy for categorical.
3. **`update_ot_transfer_matrix`** -  builds the OT cost matrix from channel/attribute
   similarity distributions and returns the transfer matrix T ∈ ℝ^{DxC}.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="enable_nested_tensor")

# ---------------------------------------------------------------------------
# FT-Transformer
# ---------------------------------------------------------------------------

class FTTransformer(nn.Module):
    """
    Lightweight FT-Transformer for tabular data.

    Outputs:
      y_pred    : (B, output_dim)   regression prediction from CLS token
      attr_embs : (B, D, embed_dim) per-attribute embeddings (for OT alignment)
    """

    def __init__(self, n_num, cat_cardinalities, embed_dim=32, n_blocks=2,
                 output_dim=5, dropout=0.1):
        super().__init__()
        self.n_num = n_num
        self.n_cat = len(cat_cardinalities)
        self.D = n_num + len(cat_cardinalities)

        # One linear tokeniser per numerical feature: scalar → embed_dim
        self.num_tokenizers = nn.ModuleList([
            nn.Linear(1, embed_dim) for _ in range(n_num)
        ])

        # One embedding table per categorical feature
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(n_c + 1, embed_dim) # +1 safety margin for unseen labels
            for n_c in cat_cardinalities
        ])

        # Learnable CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.normal_(self.cls_token, std=0.02)

        # Transformer encoder (norm_first = Pre-LN, more stable)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=4, # 32 / 4 = 8 head-dim
            dim_feedforward=4 * embed_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.norm = nn.LayerNorm(embed_dim)

        # Regression head on CLS token
        self.reg_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, output_dim),
        )

    def forward(self, x_num, x_cat):
        """
        x_num : (B, n_num) float32
        x_cat : (B, n_cat) int64
        """
        tokens = []
        for i, tok in enumerate(self.num_tokenizers):
            tokens.append(tok(x_num[:, i:i+1]).unsqueeze(1))   # (B, 1, E)
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i]).unsqueeze(1)) # (B, 1, E)

        x = torch.cat(tokens, dim=1)# (B, D, E)
        cls = self.cls_token.expand(x.shape[0], -1, -1) # (B, 1, E)
        x = torch.cat([cls, x], dim=1) # (B, D+1, E)
        x = self.transformer(x)
        x = self.norm(x)

        cls_out   = x[:, 0, :] # (B, E)
        attr_embs = x[:, 1:, :] # (B, D, E)  per-attribute embeddings
        y_pred = self.reg_head(cls_out) # (B, output_dim)
        return y_pred, attr_embs


# ---------------------------------------------------------------------------
# Li2t Heads  (image → attribute prediction)
# ---------------------------------------------------------------------------

class Li2tHeads(nn.Module):
    """
    One prediction head per tabular attribute.
    The OT transfer matrix T selects which image channels are relevant for each
    attribute; the head then predicts the attribute value from those channels.
    """

    def __init__(self, img_dim: int, n_num: int, cat_cardinalities):
        super().__init__()
        self.n_num = n_num
        self.n_cat = len(cat_cardinalities)

        # Numerical: weighted channels → scalar
        self.num_heads = nn.ModuleList([
            nn.Linear(img_dim, 1) for _ in range(n_num)
        ])
        # Categorical: weighted channels → class logits
        self.cat_heads = nn.ModuleList([
            nn.Linear(img_dim, n_c) for n_c in cat_cardinalities
        ])

    def compute_li2t_loss(self, img_feat, T, tab_num, tab_cat):
        """
        img_feat : (B, C) raw CLS image features (C=384)
        T        : (D, C) OT transfer matrix (on same device)
        tab_num  : (B, n_num) numerical targets
        tab_cat  : (B, n_cat) categorical class indices

        Implements Li2t = Σ_p MSE(T_p ⊙ φ(xI)) + Σ_q CE(T_q ⊙ φ(xI))
        where ⊙ is element-wise weighting before the head projection.
        """
        loss = img_feat.new_zeros(())

        for p in range(self.n_num):
            w    = T[p] # (C,)
            feat = img_feat * w.unsqueeze(0) # (B, C)
            pred = self.num_heads[p](feat).squeeze(-1) # (B,)
            loss = loss + nn.functional.mse_loss(pred, tab_num[:, p])

        for q in range(self.n_cat):
            w      = T[self.n_num + q]
            feat   = img_feat * w.unsqueeze(0)
            logits = self.cat_heads[q](feat) # (B, n_c)
            loss   = loss + nn.functional.cross_entropy(logits, tab_cat[:, q])

        return loss


# ---------------------------------------------------------------------------
# OT Cost-Matrix & Transfer-Matrix computation
# ---------------------------------------------------------------------------

def update_ot_transfer_matrix(
    X_img_np: np.ndarray,
    attr_embs_np: np.ndarray,
    cluster_labels: np.ndarray,
    n_clusters: int,
) -> np.ndarray:
    """
    Compute OT-based transfer matrix  T ∈ ℝ^{D x C}.

    Algorithm (CHARMS §4.2):
      1. For each cluster k: SI_k[i,j] = cos-sim of cluster-k sub-vectors.
      2. For each attribute j: ST_j[i,j] = cos-sim of FT-Transformer embeddings.
      3. Cost C[j,k] = ||SI_k - ST_j||²_F.
      4. Solve OT with uniform distributions → T_ot ∈ ℝ^{Dxk}.
      5. Map cluster weights back to individual dimensions → T_full ∈ ℝ^{DxC}.

    X_img_np    : (N, C)
    attr_embs_np: (N, D, E)
    cluster_labels: (C,)  K-Means assignment for each feature dimension
    """
    N, C = X_img_np.shape
    D    = attr_embs_np.shape[1]

    # --- Step 1: channel similarity matrices ---
    SI = np.empty((n_clusters, N, N), dtype=np.float32)
    for k in range(n_clusters):
        mask = cluster_labels == k
        if not mask.any():
            SI[k] = 0.0
            continue
        rep = X_img_np[:, mask] # (N, m_k)
        nrm = np.linalg.norm(rep, axis=1, keepdims=True) + 1e-8
        rep_n = rep / nrm
        SI[k] = rep_n @ rep_n.T # (N, N)

    # --- Step 2: attribute similarity matrices ---
    ST = np.empty((D, N, N), dtype=np.float32)
    for j in range(D):
        rep = attr_embs_np[:, j, :] # (N, E)
        nrm = np.linalg.norm(rep, axis=1, keepdims=True) + 1e-8
        rep_n = rep / nrm
        ST[j] = rep_n @ rep_n.T

    # --- Step 3: cost matrix C[j, k] = ||SI_k − ST_j||²_F ---
    C_mat = np.zeros((D, n_clusters), dtype=np.float64)
    for j in range(D):
        for k in range(n_clusters):
            diff = SI[k] - ST[j]
            C_mat[j, k] = float(np.sum(diff * diff))
    C_mat /= (C_mat.max() + 1e-8) # normalise to [0, 1]

    # --- Step 4: solve OT (exact EMD, uniform marginals) ---
    a = np.ones(D,          dtype=np.float64) / D
    b = np.ones(n_clusters, dtype=np.float64) / n_clusters
    T_ot = ot.emd(a, b, C_mat) # (D, n_clusters)

    # --- Step 5: restore cluster → individual dimension mapping ---
    T_full = np.zeros((D, C), dtype=np.float32)
    for k in range(n_clusters):
        mask = cluster_labels == k
        if mask.any():
            T_full[:, mask] = T_ot[:, k:k+1].astype(np.float32)

    # Row-normalise so each attribute's weight sums to 1
    row_sums = T_full.sum(axis=1, keepdims=True) + 1e-8
    T_full /= row_sums

    return T_full # (D, C)


# ---------------------------------------------------------------------------
# Dataset with tabular columns
# ---------------------------------------------------------------------------

class CHARMSDataset(Dataset):
    """Like BiomassDataset but also carries tabular numerical and categorical features."""

    def __init__(self, X, y, tab_num, tab_cat):
        self.X       = torch.tensor(X,       dtype=torch.float32)
        self.y       = torch.tensor(y,       dtype=torch.float32)
        self.tab_num = torch.tensor(tab_num, dtype=torch.float32)
        self.tab_cat = torch.tensor(tab_cat, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i], self.tab_num[i], self.tab_cat[i]


print("CHARMS modules defined:")
print(f"  FTTransformer  - {sum(p.numel() for p in FTTransformer(N_NUM_ATTRS, N_CAT_CLASSES, cfg.tab_embed_dim, cfg.ft_n_blocks, cfg.output_dim, cfg.dropout).parameters()):,} params")
print(f"  Li2tHeads      - {sum(p.numel() for p in Li2tHeads(cfg.input_dim, N_NUM_ATTRS, N_CAT_CLASSES).parameters()):,} params")
print(f"  OT subsample   - {cfg.ot_subsample}, update every {cfg.ot_update_freq} epochs")

2026-04-24 08:04:40.147394: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777017880.303160      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777017880.348022      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777017880.722782      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777017880.722814      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777017880.722817      16 computation_placer.cc:177] computation placer alr

CHARMS modules defined:
  FTTransformer  — 27,333 params
  Li2tHeads      — 5,775 params
  OT subsample   — 512, update every 5 epochs


## 9. Loss, Metrics & Dataset
Weighted SmoothL1, weighted global R², per-target RMSE, and a minimal PyTorch Dataset.

In [ ]:
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
_LOSS_WEIGHTS = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)


def _weighted_smooth_l1(pred, target, weights, beta=1.0):
    loss_per = nn.functional.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss_per * weights.to(pred.device)).mean()


def weighted_global_r2(y_true, y_pred):
    w  = WEIGHT_VECTOR
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


class BiomassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


@torch.no_grad()
def eval_epoch(model, loader, eval_device):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    for X, y in loader:
        X, y = X.to(eval_device), y.to(eval_device)
        pred = model(X)
        total_loss += _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS).item() * len(X)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    r2   = weighted_global_r2(y_true, y_pred)
    rmse = rmse_per_target(y_true, y_pred)
    return total_loss / len(loader.dataset), r2, rmse, y_pred, y_true


print("Loss, metrics, dataset, eval_epoch defined.")

Loss, metrics, dataset, eval_epoch defined.


## 10. Model Training - Baseline
Standard minibatch training loop with AdamW + cosine LR schedule; best checkpoint tracked by val R².

In [ ]:
def train_baseline_loop(model, train_ds, val_ds, epochs, lr, weight_decay, batch_size,
                        seed, train_device, verbose=True):
    """Standard minibatch training loop. Returns (history, best_state_dict)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_ldr = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_ldr   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr / 100)

    history = {"train_loss": [], "val_loss": [], "val_r2": [], "val_rmse": []}
    best_val_r2 = -float("inf")
    best_state  = None

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss, n_steps = 0.0, 0
        for X_b, y_b in train_ldr:
            X_b, y_b = X_b.to(train_device), y_b.to(train_device)
            optimizer.zero_grad()
            pred = model(X_b)
            loss = _weighted_smooth_l1(pred, y_b, _LOSS_WEIGHTS)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * len(X_b)
            n_steps += len(X_b)

        scheduler.step()

        tr = ep_loss / n_steps
        val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_ldr, train_device)

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(tr)
        history["val_loss"].append(val_loss)
        history["val_r2"].append(val_r2)
        history["val_rmse"].append(val_rmse)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            rmse_str = "  ".join(
                f"{t.split('_')[1]}:{v:.2f}" for t, v in val_rmse.items()
            )
            print(
                f"  ep {epoch:3d}  tr={tr:.4f}  "
                f"val_loss={val_loss:.4f}  val_R²={val_r2:.4f}  [{rmse_str}]"
            )

    return history, best_state


train_ds = BiomassDataset(X_train, Y_train_raw)
val_ds   = BiomassDataset(X_val,   Y_val_raw)

torch.manual_seed(cfg.seed)
model = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout).to(device)

history, best_state = train_baseline_loop(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=cfg.epochs,
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    batch_size=cfg.batch_size,
    seed=cfg.seed,
    train_device=device,
    verbose=True,
)

best_epoch = int(np.argmax(history["val_r2"])) + 1
print(f"\nBaseline best val R² = {max(history['val_r2']):.4f} at epoch {best_epoch}")

  ep   1  tr=6.4425  val_loss=6.7538  val_R²=-0.9497  [Green:39.57  Dead:18.28  Clover:14.13  g:43.80  Total:52.96]
  ep  10  tr=2.2938  val_loss=2.3144  val_R²=0.5505  [Green:22.55  Dead:13.27  Clover:12.90  g:20.76  Total:17.78]
  ep  20  tr=1.8943  val_loss=1.9321  val_R²=0.6712  [Green:17.54  Dead:13.01  Clover:12.28  g:15.51  Total:14.43]
  ep  30  tr=1.6576  val_loss=1.8283  val_R²=0.7133  [Green:15.57  Dead:12.62  Clover:11.43  g:13.89  Total:14.56]
  ep  40  tr=1.5550  val_loss=1.7853  val_R²=0.7384  [Green:14.81  Dead:11.87  Clover:10.66  g:13.66  Total:14.51]
  ep  50  tr=1.4546  val_loss=1.7474  val_R²=0.7567  [Green:14.36  Dead:11.25  Clover:10.18  g:13.41  Total:14.36]
  ep  60  tr=1.4433  val_loss=1.7445  val_R²=0.7646  [Green:14.09  Dead:10.88  Clover:9.96  g:13.26  Total:14.32]
  ep  70  tr=1.4212  val_loss=1.7405  val_R²=0.7670  [Green:14.02  Dead:10.76  Clover:9.88  g:13.18  Total:14.40]
  ep  80  tr=1.4149  val_loss=1.7321  val_R²=0.7693  [Green:13.90  Dead:10.73  Cl

## 10b. CHARMS Training

Full loss (Eq. 7 from the paper):

```
L = L_image  +  λ_tab · L_tabular  +  λ_li2t · Li2t
```

where:
- **L_image** - weighted SmoothL1 on biomass targets from the image model (unchanged from baseline)
- **L_tabular** - weighted SmoothL1 on biomass targets from the FT-Transformer (trains the tabular
  model so its attribute embeddings improve over time for better OT alignment)
- **Li2t** - auxiliary MSE/CE losses that force image channels to predict tabular attributes
  (channel-attribute knowledge transfer)

The OT transfer matrix T is recomputed every `ot_update_freq` epochs on a random
`ot_subsample`-size subset of training data.

In [ ]:
def train_charms_loop(
    model,
    ft_transformer,
    li2t_heads,
    train_ds, # CHARMSDataset - returns (X, y, tab_num, tab_cat)
    X_train_np, # (N, C) numpy array for OT computation
    X_tab_num_train_np, # (N, n_num)
    X_tab_cat_train_np,# (N, n_cat)
    channel_cluster_labels, # (C,)
    val_biomass_ds, # plain BiomassDataset for eval_epoch
    cfg,
    train_device,
    verbose=True,
):
    """CHARMS training loop. Returns (history, best_state_dict_for_image_model)."""
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)

    train_ldr = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0
    )
    val_ldr = DataLoader(
        val_biomass_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0
    )

    all_params = (
        list(model.parameters()) +
        list(ft_transformer.parameters()) +
        list(li2t_heads.parameters())
    )
    optimizer = torch.optim.AdamW(all_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.epochs, eta_min=cfg.lr / 100
    )

    # Initialise transfer matrix: uniform weights
    C_dim = X_train_np.shape[1]  # 384
    T_mat = torch.ones(D_TAB, C_dim, device=train_device) / C_dim

    history = {"train_loss": [], "val_loss": [], "val_r2": [], "val_rmse": []}
    best_val_r2, best_state = -float("inf"), None

    for epoch in range(1, cfg.epochs + 1):

        # ----------------------------------------------------------------
        # Refresh OT transfer matrix every ot_update_freq epochs
        # ----------------------------------------------------------------
        if epoch == 1 or epoch % cfg.ot_update_freq == 0:
            ft_transformer.eval()
            ot_n   = min(cfg.ot_subsample, len(X_train_np))
            ot_idx = np.random.choice(len(X_train_np), ot_n, replace=False)

            with torch.no_grad():
                t_num = torch.tensor(
                    X_tab_num_train_np[ot_idx], dtype=torch.float32, device=train_device
                )
                t_cat = torch.tensor(
                    X_tab_cat_train_np[ot_idx], dtype=torch.long, device=train_device
                )
                _, attr_embs = ft_transformer(t_num, t_cat)
                attr_embs_np = attr_embs.cpu().numpy()  # (ot_n, D, E)

            T_np  = update_ot_transfer_matrix(
                X_img_np       = X_train_np[ot_idx],
                attr_embs_np   = attr_embs_np,
                cluster_labels = channel_cluster_labels,
                n_clusters     = cfg.n_clusters,
            )
            T_mat = torch.tensor(T_np, dtype=torch.float32, device=train_device)

        # ----------------------------------------------------------------
        # Training step
        # ----------------------------------------------------------------
        model.train()
        ft_transformer.train()
        li2t_heads.train()

        ep_img_loss, n_steps = 0.0, 0
        for X_b, y_b, tab_num_b, tab_cat_b in train_ldr:
            X_b       = X_b.to(train_device)
            y_b       = y_b.to(train_device)
            tab_num_b = tab_num_b.to(train_device)
            tab_cat_b = tab_cat_b.to(train_device)

            optimizer.zero_grad()

            # (1) Image model
            img_pred = model(X_b)
            loss_img = _weighted_smooth_l1(img_pred, y_b, _LOSS_WEIGHTS)

            # (2) Tabular model (trains FT-Transformer for better OT embeddings)
            tab_pred, _ = ft_transformer(tab_num_b, tab_cat_b)
            loss_tab = _weighted_smooth_l1(tab_pred, y_b, _LOSS_WEIGHTS)

            # (3) Li2t auxiliary loss (channel → attribute prediction)
            loss_li2t = li2t_heads.compute_li2t_loss(
                X_b, T_mat, tab_num_b, tab_cat_b
            )

            loss = (loss_img
                    + cfg.tab_loss_weight * loss_tab
                    + cfg.li2t_weight    * loss_li2t)
            loss.backward()
            optimizer.step()

            ep_img_loss += loss_img.item() * len(X_b)
            n_steps     += len(X_b)

        scheduler.step()

        tr = ep_img_loss / n_steps
        val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_ldr, train_device)

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(tr)
        history["val_loss"].append(val_loss)
        history["val_r2"].append(val_r2)
        history["val_rmse"].append(val_rmse)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            rmse_str = "  ".join(
                f"{t.split('_')[1]}:{v:.2f}" for t, v in val_rmse.items()
            )
            print(
                f"  ep {epoch:3d}  img={tr:.4f}  "
                f"val_loss={val_loss:.4f}  val_R²={val_r2:.4f}  [{rmse_str}]"
            )

    return history, best_state


# --- Instantiate CHARMS modules ---
ft_transformer = FTTransformer(
    n_num            = N_NUM_ATTRS,
    cat_cardinalities= N_CAT_CLASSES,
    embed_dim        = cfg.tab_embed_dim,
    n_blocks         = cfg.ft_n_blocks,
    output_dim       = cfg.output_dim,
    dropout          = cfg.dropout,
).to(device)

li2t_heads = Li2tHeads(
    img_dim          = cfg.input_dim, # 384
    n_num            = N_NUM_ATTRS,
    cat_cardinalities= N_CAT_CLASSES,
).to(device)

# --- Datasets ---
charms_train_ds = CHARMSDataset(
    X_train, Y_train_raw, X_tab_num_train, X_tab_cat_train
)
val_biomass_ds = BiomassDataset(X_val, Y_val_raw) # plain, for eval_epoch

# --- Fresh image model ---
torch.manual_seed(cfg.seed)
model_charms = BiomassModel(
    cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout
).to(device)

print("Starting CHARMS training...")
history_charms, best_state_charms = train_charms_loop(
    model                 = model_charms,
    ft_transformer        = ft_transformer,
    li2t_heads            = li2t_heads,
    train_ds              = charms_train_ds,
    X_train_np            = X_train,
    X_tab_num_train_np    = X_tab_num_train,
    X_tab_cat_train_np    = X_tab_cat_train,
    channel_cluster_labels= channel_cluster_labels,
    val_biomass_ds        = val_biomass_ds,
    cfg                   = cfg,
    train_device          = device,
    verbose               = True,
)

best_epoch_charms = int(np.argmax(history_charms["val_r2"])) + 1
print(f"\nCHARMS best val R² = {max(history_charms['val_r2']):.4f} at epoch {best_epoch_charms}")
print(f"Baseline best val R² = {max(history['val_r2']):.4f}")
print(f"Delta = {max(history_charms['val_r2']) - max(history['val_r2']):+.4f}")

Starting CHARMS training...
  ep   1  img=6.4403  val_loss=6.7521  val_R²=-0.9495  [Green:39.57  Dead:18.29  Clover:14.15  g:43.81  Total:52.94]
  ep  10  img=2.2959  val_loss=2.3402  val_R²=0.5526  [Green:22.23  Dead:13.28  Clover:12.74  g:20.78  Total:18.20]
  ep  20  img=1.8482  val_loss=1.9262  val_R²=0.6784  [Green:17.45  Dead:13.03  Clover:11.96  g:15.40  Total:14.30]
  ep  30  img=1.6861  val_loss=1.8374  val_R²=0.7214  [Green:15.46  Dead:12.54  Clover:10.70  g:14.30  Total:14.45]
  ep  40  img=1.5934  val_loss=1.7575  val_R²=0.7564  [Green:14.07  Dead:11.82  Clover:9.89  g:13.46  Total:13.99]
  ep  50  img=1.5017  val_loss=1.7511  val_R²=0.7710  [Green:13.68  Dead:11.32  Clover:9.39  g:13.09  Total:14.05]
  ep  60  img=1.4859  val_loss=1.7325  val_R²=0.7761  [Green:13.54  Dead:11.03  Clover:9.21  g:13.18  Total:14.16]
  ep  70  img=1.4858  val_loss=1.7432  val_R²=0.7747  [Green:13.59  Dead:10.93  Clover:9.16  g:13.34  Total:14.40]
  ep  80  img=1.5017  val_loss=1.7424  val_R²=0

## 11. Final Validation Metrics
Evaluate best checkpoint; flag if weighted R² < 0.75.

In [ ]:
val_loader = DataLoader(val_biomass_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
_, val_r2_best, val_rmse_best, val_preds, val_true = eval_epoch(model_charms, val_loader, device)

print("=" * 55)
print(f"Final val weighted R²: {val_r2_best:.4f}")
print("=" * 55)
print("RMSE per target:")
for t, v in val_rmse_best.items():
    print(f"  {t:<18}: {v:.4f}")
print("=" * 55)

if val_r2_best < 0.75:
    print(
        f"\n  *** WARNING: val R² = {val_r2_best:.4f} is below the expected baseline."
        "\n      Check DINOv2 weights / resize / normalisation."
    )
else:
    print(f"\n  R² = {val_r2_best:.4f}")

Final val weighted R²: 0.7774
RMSE per target:
  Dry_Green_g       : 13.5221
  Dry_Dead_g        : 10.8974
  Dry_Clover_g      : 9.1260
  GDM_g             : 13.2308
  Dry_Total_g       : 14.2606

  R² = 0.7774 — looks reasonable.


## 12. Test Inference
Encode test features through encoder+head; clip negative predictions.
**Tabular data is not used here** - inference is image-only, as in the CHARMS design.

In [17]:
model_charms.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

with torch.no_grad():
    test_preds = model_charms(X_test_t).cpu().numpy()

test_preds = np.clip(test_preds, 0.0, None)

print(f"test_preds shape: {test_preds.shape}")
print(f"test_preds range: [{test_preds.min():.2f}, {test_preds.max():.2f}]")
print("Sample predictions (first 3 test images):")
for i in range(min(3, len(test_image_ids))):
    print(f"  {test_image_ids[i]}: {dict(zip(TARGETS, test_preds[i].round(2)))}")

test_preds shape: (1, 5)
test_preds range: [2.37, 57.51]
Sample predictions (first 3 test images):
  ID1001187975: {'Dry_Green_g': np.float32(34.52), 'Dry_Dead_g': np.float32(16.86), 'Dry_Clover_g': np.float32(2.37), 'GDM_g': np.float32(36.61), 'Dry_Total_g': np.float32(57.51)}


## 13. Format Predictions as Submission CSV
Map (image_id, 5 targets) → long-format rows matching test.csv sample_id keys.

In [18]:
def prepare_submission(test_csv_path, predictions, image_ids):
    df_t = pd.read_csv(test_csv_path)

    pred_dict = {
        img_id: {col: float(val) for col, val in zip(TARGETS, pred_row)}
        for img_id, pred_row in zip(image_ids, predictions)
    }

    def _get_pred(row):
        img_id      = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        return max(0.0, pred_dict.get(img_id, {}).get(target_name, 0.0))

    df_t["target"] = df_t.apply(_get_pred, axis=1)
    return df_t[["sample_id", "target"]]


submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)

out_path = os.path.join(cfg.output_dir, "submission.csv")
submission.to_csv(out_path, index=False)

print(f"Submission saved to: {out_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nTarget value stats:")
print(submission["target"].describe().round(3))
print("\nWorking dir:", os.listdir(cfg.output_dir))

Submission saved to: /kaggle/working/submission.csv
Rows: 5
                    sample_id     target
0  ID1001187975__Dry_Clover_g   2.368664
1    ID1001187975__Dry_Dead_g  16.859755
2   ID1001187975__Dry_Green_g  34.521023
3   ID1001187975__Dry_Total_g  57.507320
4         ID1001187975__GDM_g  36.605972

Target value stats:
count     5.000
mean     29.573
std      20.952
min       2.369
25%      16.860
50%      34.521
75%      36.606
max      57.507
Name: target, dtype: float64

Working dir: ['__notebook__.ipynb', 'submission.csv']


#### Code done

---

### 14.1 Styling & helpers
Used AI tools to visualize plots and tables.

In [ ]:
# --- Publication figure styling -------------------------------------------
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

FIG_DIR = Path(cfg.output_dir) / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# A restrained, paper-friendly rcParams setup. Serif fonts match most LaTeX
# templates; vector PDFs are the primary output, PNG as a convenience.
mpl.rcParams.update({
    "figure.dpi":         110,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "font.family":        "serif",
    "font.size":          10,
    "axes.titlesize":     11,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.25,
    "grid.linewidth":     0.5,
    "legend.frameon":     False,
    "legend.fontsize":    9,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "lines.linewidth":    1.5,
})

# Colour-blind friendly palette (Wong / Okabe–Ito).
C_BASELINE = "#0072B2"   # blue
C_CHARMS   = "#D55E00"   # vermillion
C_NEUTRAL  = "#555555"

def _save(fig, name):
    """Save a figure as both PDF (for LaTeX) and PNG (for quick viewing)."""
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"  saved: {name}.pdf / .png")

print(f"Figure directory: {FIG_DIR}")

### 14.2 Training curves (Fig. 1)
Validation weighted R² and loss over epochs. The baseline/CHARMS comparison is the main takeaway; best-epoch markers make the delta easy to read.

In [ ]:
# --- Figure 1: Training curves (baseline vs CHARMS) -----------------------
# Two side-by-side panels: validation weighted R² and validation loss.
# Best-epoch markers make the improvement easy to read off the plot.

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharex=True)
epochs_ax = np.arange(1, len(history["val_r2"]) + 1)

# --- Panel (a): validation weighted R² ---
ax = axes[0]
ax.plot(epochs_ax, history["val_r2"],
        color=C_BASELINE, label="Baseline")
ax.plot(epochs_ax, history_charms["val_r2"],
        color=C_CHARMS,  label="CHARMS")

# Mark the best epoch for each model.
be_base   = int(np.argmax(history["val_r2"])) + 1
be_charms = int(np.argmax(history_charms["val_r2"])) + 1
ax.scatter([be_base],   [max(history["val_r2"])],
           color=C_BASELINE, s=28, zorder=5,
           label=f"Best baseline (ep {be_base})")
ax.scatter([be_charms], [max(history_charms["val_r2"])],
           color=C_CHARMS,  s=28, zorder=5,
           label=f"Best CHARMS (ep {be_charms})")
ax.set_xlabel("Epoch")
ax.set_ylabel(r"Validation weighted $R^2$")
ax.set_title("(a) Validation $R^2$")
ax.legend(loc="lower right")

# --- Panel (b): validation loss (log scale — loss often spans an order of magnitude) ---
ax = axes[1]
ax.plot(epochs_ax, history["val_loss"],
        color=C_BASELINE, label="Baseline")
ax.plot(epochs_ax, history_charms["val_loss"],
        color=C_CHARMS,  label="CHARMS")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss (weighted SmoothL1)")
ax.set_yscale("log")
ax.set_title("(b) Validation loss")
ax.legend(loc="upper right")

fig.tight_layout()
_save(fig, "fig1_training_curves")
plt.show()